# Lab 3: a re-runnable ingester

Team E. The ingester lives in the team repository at `scripts/ingest.py`. This notebook clones the repository and runs it, which also proves it works on a machine that has never seen it before.

Repository: https://github.com/zukazudo/dsai6226-data-engineering

In [ ]:
# Close any DuckDB handle an earlier run of this notebook left open.
# DuckDB allows a single writer, and the ingester runs in its own process,
# so a live connection here blocks it.
try:
    con.close()
except NameError:
    pass

%cd /content
!rm -rf dsai6226-data-engineering
!git clone -q https://github.com/zukazudo/dsai6226-data-engineering.git
%cd /content/dsai6226-data-engineering
!pip install -q duckdb

/content
/content/dsai6226-data-engineering


**Observation**

Nothing is built yet. The repository holds the source CSV, the SQL, and the ingester. There is no database file, because the warehouse is generated and deliberately not committed.

In [ ]:
!python scripts/ingest.py

09:07:34  INFO    NumExpr defaulting to 2 threads.
09:07:34  INFO    run 1: adult.csv (4,104,734 bytes, sha256 250e154ed757)
09:07:37  INFO      rows read 32561
09:07:37  INFO      staged    32561  (0 already present from an earlier run)
09:07:37  INFO      rejected     0
09:07:37  INFO      loaded    32561
09:07:37  INFO      fact_person now holds 32,561 rows  (3812 ms)

load history
------------------------------------------------------------------------------
 run  source                          read  staged  rejected  loaded  status
   1  adult.csv                      32561   32561         0   32561  ok

table sizes
------------------------------------------------------------------------------
  dim_education                  16
  dim_marital_status              7
  dim_native_country             42
  dim_occupation                 16
  dim_race                        5
  dim_relationship                6
  dim_sex                         2
  dim_workclass                   9
  f

**Observation**

One command. It created the schema, landed 32,561 rows in staging, rejected none, and loaded all of them into `fact_person`.

The run records the sha256 and byte count of the file it read, so the load can later be tied to a specific version of a specific file.

In [ ]:
!python scripts/ingest.py

09:07:38  INFO    NumExpr defaulting to 2 threads.
09:07:38  INFO    run 2: adult.csv (4,104,734 bytes, sha256 250e154ed757)
09:07:39  INFO      rows read 32561
09:07:39  INFO      staged        0  (32561 already present from an earlier run)
09:07:39  INFO      rejected     0
09:07:39  INFO      loaded        0
09:07:39  INFO      fact_person now holds 32,561 rows  (1250 ms)

load history
------------------------------------------------------------------------------
 run  source                          read  staged  rejected  loaded  status
   1  adult.csv                      32561   32561         0   32561  ok
   2  adult.csv                      32561       0         0       0  ok

table sizes
------------------------------------------------------------------------------
  dim_education                  16
  dim_marital_status              7
  dim_native_country             42
  dim_occupation                 16
  dim_race                        5
  dim_relationship                

**Observation: this is the idempotency proof**

The second run read the same 32,561 rows and loaded **0**. `fact_person` still holds 32,561. Nothing was duplicated and nothing was destroyed.

This works because the source has no identifier, so the ingester manufactures one. `record_key` is an md5 of all 15 source fields plus the occurrence number of that content within the file. Re-reading the file reproduces exactly the same keys, and the insert skips every key it already holds.

The occurrence suffix matters. Three identical duplicate records become `#1`, `#2` and `#3` rather than collapsing into a single row, which is how the retain-and-mark decision from Lab 2 survives a re-run.

In [ ]:
import duckdb

DB = "warehouse/adult.duckdb"

def q(sql):
    """Open, read, close.

    The handle is never held across cells. The ingester runs in its own
    process and needs the write lock, and DuckDB allows only one writer.
    A connection left open here blocks it.
    """
    con = duckdb.connect(DB, read_only=True)
    try:
        return con.execute(sql).df()
    finally:
        con.close()

print("load history, which is the proof that outlives the terminal:")
print(q("""
    SELECT load_run_id, source_file, rows_read, rows_staged,
           rows_rejected, rows_loaded, status
    FROM load_run ORDER BY load_run_id
""").to_string(index=False))

print()
print("a record_key looks like this:")
print(q("SELECT record_key FROM fact_person LIMIT 3").to_string(index=False))

load history, which is the proof that outlives the terminal:
 load_run_id source_file  rows_read  rows_staged  rows_rejected  rows_loaded status
           1   adult.csv      32561        32561              0        32561     ok
           2   adult.csv      32561            0              0            0     ok

a record_key looks like this:
                        record_key
9d70fe2b29b2d26e5b188557b9b22c84#1
e7bd93ece8ea320761a569e610a07776#1
33eeacc81aebbd3b9f007d2c5db6c1de#1


**Observation**

Every invocation writes a row to `load_run`, so the proof survives the scrollback. Run 1 loaded 32,561 and run 2 loaded 0, against identical input.

A `record_key` is 32 hex characters of content hash, then `#`, then the occurrence number.

The helper opens the database, reads, and closes immediately. DuckDB allows a single writer, and the ingester runs in its own process, so a connection left open in the notebook would block the next ingestion.

In [ ]:
# A nine row file built to trip every validation rule once.
!cat tests/fixtures/adult_bad_rows.csv
print()
!python scripts/ingest.py tests/fixtures/adult_bad_rows.csv

"age","workclass","fnlwgt","education","education.num","marital.status","occupation","relationship","race","sex","capital.gain","capital.loss","hours.per.week","native.country","income"
39,"State-gov",999001,"Bachelors",13,"Never-married","Adm-clerical","Not-in-family","White","Male",2174,0,40,"United-States","<=50K"
"abc","Private",215646,"HS-grad",9,"Divorced","Handlers-cleaners","Not-in-family","White","Male",0,0,40,"United-States","<=50K"
201,"Private",234721,"11th",7,"Married-civ-spouse","Handlers-cleaners","Husband","Black","Male",0,0,40,"United-States","<=50K"
28,"Private",-5,"Bachelors",13,"Married-civ-spouse","Prof-specialty","Wife","Black","Female",0,0,40,"Cuba","<=50K"
37,"Private",284582,"HS-grad",12,"Married-civ-spouse","Exec-managerial","Wife","White","Female",0,0,40,"United-States","<=50K"
49,"Private",160187,"9th",5,"Married-spouse-absent","Other-service","Not-in-family","Black","Female",0,0,0,"Jamaica","<=50K"
52,"Self-emp-not-inc",209642,"HS-grad",9,"Married-civ-spous

**Observation**

Nine rows in, one loaded, eight rejected, each for a different reason. The reject path is demonstrated rather than assumed, which matters because on the real file it never fires.

The rules reject rows that are **malformed**, not rows that are merely **odd**. The 2,399 records carrying `?` and the three records contradicting themselves on sex and relationship are real observations and load normally. Rejecting them would be throwing away data because it is inconvenient.

In [ ]:
print(q("""
    SELECT reason, count(*) AS rows, min(detail) AS example
    FROM load_reject GROUP BY reason ORDER BY reason
""").to_string(index=False))

                       reason  rows                                 example
           age_not_an_integer     1                               age = abc
             age_out_of_range     1                               age = 201
        capital_value_invalid     1                   gain = -100, loss = 0
   education_mapping_mismatch     1 education = HS-grad, education.num = 12
fnlwgt_not_a_positive_integer     1                             fnlwgt = -5
  hours_per_week_out_of_range     1                      hours.per.week = 0
        income_not_recognised     1                           income = 50K+
           sex_not_recognised     1                           sex = Unknown


**Observation**

Rejects are a table, not just a log line. A rejected row can be investigated after the run, which is the difference between a pipeline that tells you something went wrong and one that tells you what.

`education_mapping_mismatch` is the rule worth noting. `dim_education` is seeded with the canonical 16 level ladder instead of being inferred from whatever arrives, so it can act as the reference an incoming pair is checked against. A file claiming `HS-grad` at level 12 is rejected instead of quietly widening the dimension.

In [ ]:
# Lineage: take a modelled row and walk it back to the line that produced it.
print(q("""
    SELECT f.person_sk, f.source_file, f.source_row, f.load_run_id,
           r.source_sha256[1:12] AS file_sha, s.age, s.occupation, s.income
    FROM fact_person f
    JOIN load_run  r ON r.load_run_id = f.load_run_id
    JOIN stg_adult s ON s.record_key  = f.record_key
    WHERE f.duplicate_group_id IS NOT NULL
    ORDER BY f.duplicate_group_id, f.duplicate_seq
    LIMIT 6
""").to_string(index=False))

 person_sk source_file  source_row  load_run_id     file_sha age    occupation income
      6228   adult.csv        6228            1 250e154ed757  90 Other-service  <=50K
      8646   adult.csv        8646            1 250e154ed757  90 Other-service  <=50K
      7616   adult.csv        7616            1 250e154ed757  19 Other-service  <=50K
     32066   adult.csv       32066            1 250e154ed757  19 Other-service  <=50K
      7979   adult.csv        7979            1 250e154ed757  25  Craft-repair  <=50K
      8454   adult.csv        8454            1 250e154ed757  25  Craft-repair  <=50K


**Observation**

Any row in the warehouse can be traced back to the exact file, the exact line, and the exact run that admitted it, with the sha256 of the file as it was read.

Lab 1 concluded that a decision spending public money had no auditable lineage. This is the part that closes that finding.

The rows shown are duplicate group members, which is also where lineage earns its keep: two rows with identical content are distinguishable only by where they came from.

## What this closes, and what it does not

Lab 1 found three defects. The ingester addresses the third and the consequences that followed from it:

- **No primary key.** `record_key` manufactures identity from content, so incremental loading is possible and re-running is safe.
- **No incremental refresh.** Lab 1 said every load must be full replace. It no longer must be.
- **No auditable lineage.** `load_run` plus `source_file` and `source_row` on every fact row.

What it does not fix, stated plainly: two genuinely different people who match on all 15 attributes are still indistinguishable. The ingester inherits that ambiguity from the source and does not pretend to resolve it. A real fix needs a real identifier, which only the data collector can supply.